# NutriVision — Phase 1 (CT) : compilation du jeu de données et entraînement**Continuous Training pipeline, executed in a browser. No local installation.**This notebook runs the four `dvc.yaml` stages — `compile`, `validate`, `train`, `export` —against FoodSeg103, and produces a `yolov8n-seg.onnx` that replaces the stock COCO modelcurrently deployed on Lightsail.| | ||---|---|| Runtime required | GPU (T4) — `Runtime → Change runtime type → T4 GPU` || Expected duration | 20 min setup · 4–6 h training · 5 min export || Repository | `Zen-Daitsu/Nutrivision` || Output | `backend/models/yolov8n-seg.onnx` + `.classes.json` |**Before starting:** `Runtime → Change runtime type → T4 GPU`. Training on CPU is not viable.Colab disconnects idle sessions, so keep the tab active or the run is lost.

## 0 — Verify the GPU

In [ ]:
!nvidia-smiimport torchprint()print("torch      :", torch.__version__)print("cuda avail :", torch.cuda.is_available())if torch.cuda.is_available():    print("device     :", torch.cuda.get_device_name(0))else:    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 1 — Clone the repositoryA public repo needs no token. If you made it private, generate a fine-grained PAT with*Contents: read* and paste it when prompted — it is not stored in the notebook.

In [ ]:
REPO = "Zen-Daitsu/Nutrivision"PRIVATE = False   # set True if the repository is privateimport os, getpass, subprocessif os.path.exists("/content/Nutrivision"):    !rm -rf /content/Nutrivisionif PRIVATE:    token = getpass.getpass("GitHub PAT (Contents: read): ")    url = f"https://{token}@github.com/{REPO}.git"else:    url = f"https://github.com/{REPO}.git"subprocess.run(["git", "clone", "--depth", "1", url, "/content/Nutrivision"], check=True)%cd /content/Nutrivision!ls -la

## 2 — Install dependencies

In [ ]:
!pip install -q ultralytics==8.3.55 dvc[s3]==3.59.0 onnx==1.17.0 onnxslim==0.1.42 \                onnxruntime==1.20.1 opencv-python-headless==4.10.0.84 pyyamlimport ultralytics, cv2, yamlprint("ultralytics:", ultralytics.__version__)print("opencv     :", cv2.__version__)

## 3 — Obtain FoodSeg103Two routes. **A** pulls the DVC-tracked dataset from S3 and needs AWS credentials.**B** reads it from your Google Drive and needs none — preferred, since it avoidscreating an access key.Expected layout after either route:```data/FoodSeg103/Images/img_dir/train/*.jpgdata/FoodSeg103/Images/ann_dir/train/*.pngdata/FoodSeg103/category_id.txt```

### Route A — DVC pull from S3

In [ ]:
# Skip this cell entirely if using Google Drive.USE_DVC = Falseif USE_DVC:    import getpass, os    os.environ["AWS_ACCESS_KEY_ID"]     = getpass.getpass("AWS_ACCESS_KEY_ID: ")    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")    os.environ["AWS_DEFAULT_REGION"]    = "ca-central-1"    !dvc remote list    !dvc pull -v    # Revoke this key in IAM as soon as the notebook finishes.

### Route B — Google Drive (recommended)

In [ ]:
USE_DRIVE = Trueif USE_DRIVE:    from google.colab import drive    drive.mount('/content/drive')    # Adjust to wherever FoodSeg103 sits in your Drive.    SRC = "/content/drive/MyDrive/Nutrivision/FoodSeg103"    import os, shutil    assert os.path.isdir(SRC), f"Not found: {SRC}"    os.makedirs("data", exist_ok=True)    if not os.path.exists("data/FoodSeg103"):        os.symlink(SRC, "data/FoodSeg103")   # symlink, not copy: 1.4 GB stays put!ls data/FoodSeg103!echo "--- train images ---" && ls data/FoodSeg103/Images/img_dir/train | head -3!echo "--- train masks  ---" && ls data/FoodSeg103/Images/ann_dir/train | head -3

## 4 — Correct the class map**This is the step that determines whether the model learns anything correct.**`ml/class_map.yaml` maps FoodSeg103 category ids to project class ids. The ids currentlycommitted are placeholders. A wrong id trains the model on the wrong ingredient silently —`validate_dataset.py` cannot detect it, because the labels are structurally valid.Read the real category list below, then edit the mapping in the next cell.

In [ ]:
CAT_FILE = "data/FoodSeg103/category_id.txt"import osif os.path.exists(CAT_FILE):    with open(CAT_FILE) as f:        for line in f:            print(line.rstrip())else:    print(f"{CAT_FILE} not found. Look for a category list in:")    !find data/FoodSeg103 -maxdepth 2 -name "*.txt" -o -maxdepth 2 -name "*.json" | head

In [ ]:
# Edit source_to_target using the category ids printed above.# Key   = FoodSeg103 category id# Value = NutriVision class id (0-9)CLASS_MAP = {    "target_classes": {        0: "chicken_breast",        1: "beef",        2: "egg",        3: "white_rice",        4: "quinoa",        5: "broccoli",        6: "spinach",        7: "tomato",        8: "avocado",        9: "blueberry",    },    "source_to_target": {        76: 0,   # <-- VERIFY every one of these against the output above        74: 1,        72: 2,        53: 3,        56: 4,        33: 5,        35: 6,        46: 7,        22: 8,        10: 9,    },}import yamlwith open("ml/class_map.yaml", "w") as f:    yaml.safe_dump(CLASS_MAP, f, sort_keys=False, allow_unicode=True)print(open("ml/class_map.yaml").read())

## 5 — Compile the datasetConverts semantic masks to YOLO polygons. Splits deterministically on `sha1(basename)`,so the partition is identical on every machine and every rerun — the property the firstimplementation lacked, which leaked validation data into training across DVC versions.

In [ ]:
!python ml/compile_dataset.py --src data/FoodSeg103 --dst my_datasetimport jsonprint()print(open("my_dataset/compile_report.json").read())

## 6 — Data contract gateFails the run on image/label orphans, out-of-range class ids, denormalised coordinates,malformed polygons, and class starvation below 50 instances.**A non-zero exit here means stop.** Training on a dataset that fails its contract wastesfive GPU-hours and produces a model that is confidently wrong.

In [ ]:
import subprocessr = subprocess.run(["python", "ml/validate_dataset.py", "--dataset", "my_dataset"])if r.returncode != 0:    raise SystemExit("DATA CONTRACT FAILED — fix the mapping in step 4 before training.")print("\nContract satisfied. Safe to train.")

### Visual sanity checkA contract can pass while the polygons are nonsense. Look at three.

In [ ]:
import cv2, numpy as np, random, globimport matplotlib.pyplot as pltnames = yaml.safe_load(open("my_dataset/data.yaml"))["names"]samples = random.sample(glob.glob("my_dataset/train/images/*"), 3)fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, img_path in zip(axes, samples):    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)    h, w = img.shape[:2]    label = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"    for line in open(label):        parts = line.split()        cls = int(parts[0])        pts = np.array([float(v) for v in parts[1:]]).reshape(-1, 2) * [w, h]        cv2.polylines(img, [pts.astype(np.int32)], True, (255, 180, 84), 3)        cv2.putText(img, names[cls], tuple(pts[0].astype(int)),                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 180, 84), 2)    ax.imshow(img); ax.axis("off"); ax.set_title(img_path.split("/")[-1], fontsize=9)plt.tight_layout(); plt.show()

## 7 — Train`yolov8n-seg` at 640 px, 120 epochs, roughly 4–6 h on a T4 for a 10-class subset.Nano rather than small: the production node is 1 vCPU / 1 GB, where `yolov8s-seg`roughly triples inference latency for a marginal accuracy gain.Colab disconnects idle sessions. Keep the tab visible.

In [ ]:
EPOCHS = 120BATCH  = 16!python ml/train.py \    --data my_dataset/data.yaml \    --weights yolov8n-seg.pt \    --epochs {EPOCHS} \    --batch {BATCH} \    --imgsz 640 \    --project runs \    --name nutrivision

In [ ]:
import json, osif os.path.exists("runs/metrics.json"):    m = json.load(open("runs/metrics.json"))    print(json.dumps(m, indent=2))    print()    print("mask mAP50    :", round(m["map50_mask"], 4))    print("mask mAP50-95 :", round(m["map50_95_mask"], 4))    print()    print("Below ~0.30 mAP50 on masks, the model is not demo-ready.")    print("Usual causes: wrong class map, too few instances per class,")    print("or FoodSeg103 alone without the 300 real-capture images.")

## 8 — Export to ONNXRuns the PyTorch graph and the exported ONNX graph on the same tensor and asserts a maximumabsolute difference below `1e-3`. An export that loads is not an export that is correct.

In [ ]:
!python ml/export_model.py \    --weights runs/nutrivision/weights/best.pt \    --imgsz 640 \    --opset 17 \    --out backend/models/yolov8n-seg.onnx!ls -lh backend/models/

## 9 — Retrieve the artifactsSave `best.pt`, the ONNX graph and `classes.json` to Drive. Losing them to a Colabdisconnect means repeating the training run.

In [ ]:
import shutil, osDEST = "/content/drive/MyDrive/Nutrivision/artifacts"os.makedirs(DEST, exist_ok=True)for src in [    "runs/nutrivision/weights/best.pt",    "backend/models/yolov8n-seg.onnx",    "backend/models/yolov8n-seg.classes.json",    "runs/metrics.json",    "my_dataset/compile_report.json",]:    if os.path.exists(src):        shutil.copy2(src, DEST)        print("saved", src)!ls -lh {DEST}

## 10 — Ship itThe deployed container builds its ONNX graph from stock COCO weights inside`backend/Dockerfile`. To serve this model instead:1. Upload `yolov8n-seg.onnx` and `yolov8n-seg.classes.json` from Drive to   `s3://<bucket>/models/` in the AWS console.2. Edit `backend/Dockerfile` — delete the `exporter` stage and replace the two   `COPY --from=exporter` lines with a fetch from S3 in `deploy-lightsail.yml`,   as `deploy-backend.yml` already demonstrates.3. Push to `main`. The workflow rebuilds and rolls the container.4. Confirm the classes at `/healthz`, then scan a plate of chicken and rice.Once this succeeds, slide 8's "Phase 1 : 100 % Terminé" becomes true rather than aspirational.